<a href="https://colab.research.google.com/github/OutisAyo/council-classifier-/blob/main/notebooks/02_data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Machine Learning for Automated Classification and Prioritisation of Local Council Service Requests in the UK
## Notebook 02 — Data Preprocessing

**Author:** Fashina Fuad Ayomide  
**MSc Data Science, University of South Wales**

This notebook prepares the Swansea Council dataset for machine learning by cleaning missing values, dropping under-represented classes, combining text features, deriving the priority label, and splitting the data into training and test sets.

## Mounting Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Importing the libraries

In [2]:
import numpy as np
import pandas as pd

## Importing the dataset

In [3]:
dataset = pd.read_csv('/content/drive/MyDrive/council-classifier/swansea_council_foi_data.csv')
print("Original dataset shape:", dataset.shape)
dataset.head()

Original dataset shape: (260286, 7)


,source_year,category_code,complaint_type_code,assigned_department,target_date,date_received,date_closed
0,2022,Live Rat in House * Urgent*,Animal Warden - Out of Hours,Pollution Division,03/01/2022,01/01/2022,01/01/2022
1,2022,Recycling,Multiple Bag & Liner Request,Waste Management,10/01/2022,01/01/2022,05/06/2026
2,2022,Recycling,Multiple Bag & Liner Request,Waste Management,10/01/2022,01/01/2022,05/06/2026
3,2022,Domestic,Bulk Request,Waste Management,10/01/2022,01/01/2022,10/01/2022
4,2022,"General Rubbish/hous,building",NaN,Housing Division,10/01/2022,01/01/2022,06/01/2022


## Dropping under-represented classes

Administration (37 records) and Building Control (12 records) have insufficient data for reliable classification. We remove them from the dataset.

In [4]:
classes_to_drop = ['Administration', 'Building Control']
dataset = dataset[~dataset['assigned_department'].isin(classes_to_drop)]

print("Dataset shape after dropping under-represented classes:", dataset.shape)
print("\nRemaining department counts:")
print(dataset['assigned_department'].value_counts())

Dataset shape after dropping under-represented classes: (260237, 7)

Remaining department counts:
assigned_department
Waste Management              197555
Pollution Division             36262
Housing Division               10649
Licensing Division              8685
Food and Safety Division        6020
Trading Standards Division      1066
Name: count, dtype: int64


## Handling missing values in text fields

Complaint type has some missing values. We fill them with an empty string so they can be combined with category codes without errors.

In [5]:
print("Missing values before cleaning:")
print(dataset[['category_code', 'complaint_type_code']].isnull().sum())

dataset['complaint_type_code'] = dataset['complaint_type_code'].fillna('')
dataset['category_code'] = dataset['category_code'].fillna('')

print("\nMissing values after cleaning:")
print(dataset[['category_code', 'complaint_type_code']].isnull().sum())

Missing values before cleaning:
category_code             32
complaint_type_code    16193
dtype: int64

Missing values after cleaning:
category_code          0
complaint_type_code    0
dtype: int64


## Combining category and complaint into a single text feature

The classifier will use the combined text as its input. Joining both fields gives the model more information about each request.

In [11]:
dataset['request_text'] = dataset['category_code'] + ' ' + dataset['complaint_type_code']
dataset['request_text'] = dataset['request_text'].str.strip()

print("Sample combined request texts:")
print(dataset[['request_text', 'assigned_department']].head(10))

Sample combined request texts:
                                        request_text assigned_department
0  Live Rat in House * Urgent* Animal Warden - Ou...  Pollution Division
1             Recycling Multiple Bag & Liner Request    Waste Management
2             Recycling Multiple Bag & Liner Request    Waste Management
3                              Domestic Bulk Request    Waste Management
4                      General Rubbish/hous,building    Housing Division
5                Recycling Green Week non collection    Waste Management
6                Recycling Green Week non collection    Waste Management
7                              Domestic Bulk Request    Waste Management
8                 Recycling Pink Week non collection    Waste Management
9                Recycling Green Week non collection    Waste Management


## Deriving the priority label

We use the difference between date_received and date_closed to assign a priority level:
- **HIGH:** closed within 1 day
- **MEDIUM:** closed within 2-7 days
- **LOW:** took more than 7 days

In [12]:
dataset['date_received'] = pd.to_datetime(dataset['date_received'], format='%d/%m/%Y', errors='coerce')
dataset['date_closed'] = pd.to_datetime(dataset['date_closed'], format='%d/%m/%Y', errors='coerce')

dataset['days_to_close'] = (dataset['date_closed'] - dataset['date_received']).dt.days

def assign_priority(days):
    if pd.isnull(days) or days < 0:
        return None
    elif days <= 1:
        return 'HIGH'
    elif days <= 7:
        return 'MEDIUM'
    else:
        return 'LOW'

dataset['priority'] = dataset['days_to_close'].apply(assign_priority)

print("Priority distribution:")
print(dataset['priority'].value_counts())
print("\nPercentage per priority:")
print(dataset['priority'].value_counts(normalize=True) * 100)

Priority distribution:
priority
LOW       120329
MEDIUM     69006
HIGH       65042
Name: count, dtype: int64

Percentage per priority:
priority
LOW       47.303412
MEDIUM    27.127453
HIGH      25.569136
Name: proportion, dtype: float64


## Dropping rows without valid priority labels

Rows with missing or invalid dates cannot be used for priority prediction and are dropped.

In [13]:
print("Rows before dropping invalid priorities:", len(dataset))
dataset = dataset.dropna(subset=['priority'])
print("Rows after dropping invalid priorities:", len(dataset))

Rows before dropping invalid priorities: 260237
Rows after dropping invalid priorities: 254377


## Preparing features and targets

We now have two prediction tasks:
- **Task 1:** Predict `assigned_department` from `request_text`
- **Task 2:** Predict `priority` from `request_text`

Both use the same input (X) but different targets (y).

In [14]:
X = dataset['request_text'].values
y_department = dataset['assigned_department'].values
y_priority = dataset['priority'].values

print("Number of samples:", len(X))
print("Number of department classes:", len(np.unique(y_department)))
print("Number of priority classes:", len(np.unique(y_priority)))

Number of samples: 254377
Number of department classes: 6
Number of priority classes: 3


## Splitting the dataset into the Training set and Test set

We use stratified splitting to preserve the class distribution across train and test — critical when classes are imbalanced.

In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_dept_train, y_dept_test, y_prio_train, y_prio_test = train_test_split(
    X, y_department, y_priority,
    test_size=0.2,
    random_state=0,
    stratify=y_department
)

print("Training set size:", len(X_train))
print("Test set size:", len(X_test))
print("\nDepartment distribution in training set:")
print(pd.Series(y_dept_train).value_counts(normalize=True) * 100)

Training set size: 203501
Test set size: 50876

Department distribution in training set:
Waste Management              77.437457
Pollution Division            12.587162
Housing Division               4.134132
Licensing Division             3.154776
Food and Safety Division       2.324804
Trading Standards Division     0.361669
Name: proportion, dtype: float64


## Saving the processed data

We save the cleaned dataset and the train/test splits so the next notebook can load them directly.

In [17]:
import os
processed_dir = '/content/drive/MyDrive/council-classifier/processed'
os.makedirs(processed_dir, exist_ok=True)

dataset.to_csv(f'{processed_dir}/swansea_cleaned.csv', index=False)

np.save(f'{processed_dir}/X_train.npy', X_train)
np.save(f'{processed_dir}/X_test.npy', X_test)
np.save(f'{processed_dir}/y_dept_train.npy', y_dept_train)
np.save(f'{processed_dir}/y_dept_test.npy', y_dept_test)
np.save(f'{processed_dir}/y_prio_train.npy', y_prio_train)
np.save(f'{processed_dir}/y_prio_test.npy', y_prio_test)

print("Processed data saved to:", processed_dir)

Processed data saved to: /content/drive/MyDrive/council-classifier/processed
